# TripAdvisor Hotel Reviews: Preprocessing Walkthrough

Step-by-step cleaning of real hotel reviews before modeling.

**Dataset:** `../../datasets/tripadvisor_hotel_reviews.csv` (see `01-nlp-fundamentals` for theory).


In [ ]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

from nlp_helpers import DATASETS_DIR, download_nltk_data

download_nltk_data()
ps = PorterStemmer()
lemmatizer = WordNetLemmatizer()
print('Setup complete.')


## 1. Load and explore the data


In [ ]:
DATA_PATH = f'{DATASETS_DIR}/tripadvisor_hotel_reviews.csv'
data = pd.read_csv(DATA_PATH)

print('Shape:', data.shape)
print('\nColumns:', data.columns.tolist())
data.info()
print('\nRating distribution:')
print(data['Rating'].value_counts().sort_index())
data.head()


In [ ]:
# One full review (raw text) — notice lowercase style, commas, and star ratings like 4*
print(data.loc[0, 'Review'])


## 2. Lowercasing

Normalization so `Hotel` and `hotel` count as the same token.


In [ ]:
data['review_lowercase'] = data['Review'].str.lower()
data[['Rating', 'Review', 'review_lowercase']].head(2)


## 3. Stopword removal

Remove high-frequency words (`the`, `is`, `at`) that rarely help classification.
We **keep `not`** because it flips meaning (*not good* vs *good*).


In [ ]:
en_stopwords = set(stopwords.words('english'))
en_stopwords.discard('not')

def remove_stopwords(text):
    return ' '.join(w for w in text.split() if w not in en_stopwords)

data['review_no_stopwords'] = data['review_lowercase'].apply(remove_stopwords)
print('Before:', data.loc[0, 'review_lowercase'][:120], '...')
print('\nAfter: ', data.loc[0, 'review_no_stopwords'][:120], '...')


## 4. Punctuation and symbols

Reviews use `*` for star ratings (e.g. `4*`). We replace `*` with the word `star` before stripping other punctuation.
Then remove remaining non-word characters with a regex.


In [ ]:
def clean_punctuation(text):
    text = re.sub(r'\*', 'star', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

data['review_clean'] = data['review_no_stopwords'].apply(clean_punctuation)
print(data.loc[2, 'review_no_stopwords'][:80], '...')  # has 4*
print(data.loc[2, 'review_clean'][:80], '...')


## 5. Tokenization

Split text into individual tokens. NLTK's `word_tokenize` handles edge cases better than `str.split()`.


In [ ]:
data['tokenized'] = data['review_clean'].apply(word_tokenize)
print('Token count (review 0):', len(data.loc[0, 'tokenized']))
print('First 20 tokens:', data.loc[0, 'tokenized'][:20])


## 6. Stemming

Rule-based suffix stripping (fast, can produce non-words like `servic` from `service`).


In [ ]:
def stem_tokens(tokens):
    return [ps.stem(t) for t in tokens]

data['stemmed'] = data['tokenized'].apply(stem_tokens)
pair = list(zip(data.loc[0, 'tokenized'][:8], data.loc[0, 'stemmed'][:8]))
print('token -> stem')
for tok, stem in pair:
    print(f'  {tok:12} -> {stem}')


## 7. Lemmatization

Maps tokens to dictionary forms (slower, more readable than stemming).
Default lemmatizer assumes nouns; stemming is shown above for comparison on the same tokens.


In [ ]:
def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(t) for t in tokens]

data['lemmatized'] = data['tokenized'].apply(lemmatize_tokens)
pair = list(zip(data.loc[0, 'tokenized'][:8], data.loc[0, 'stemmed'][:8], data.loc[0, 'lemmatized'][:8]))
print(f"{'token':12} {'stem':12} lemma")
for tok, stem, lemma in pair:
    print(f'  {tok:12} {stem:12} {lemma}')


## 8. N-grams

Sequences of *n* adjacent tokens capture phrases (e.g. `customer service` as a bigram).
We build n-grams from **lemmatized** tokens across the whole corpus and count the most common.


In [ ]:
tokens_clean = sum(data['lemmatized'], [])

unigrams = pd.Series(nltk.ngrams(tokens_clean, 1)).value_counts().head(10)
bigrams = pd.Series(nltk.ngrams(tokens_clean, 2)).value_counts().head(10)

print('Top 10 unigrams:')
print(unigrams)
print('\nTop 10 bigrams:')
print(bigrams)


In [ ]:
four_grams = pd.Series(nltk.ngrams(tokens_clean, 4)).value_counts().head(5)
print('Top 5 four-grams (longer phrases):')
print(four_grams)


## Summary

| Step | Column | Purpose |
|------|--------|---------|
| Lowercase | `review_lowercase` | Normalize case |
| Stopwords | `review_no_stopwords` | Drop noise; keep `not` |
| Punctuation | `review_clean` | `*` → `star`, strip symbols |
| Tokenize | `tokenized` | List of words |
| Stem | `stemmed` | Aggressive normalization |
| Lemmatize | `lemmatized` | Readable base forms |
| N-grams | — | Phrase-level features |

Next: feed `lemmatized` or joined strings into `CountVectorizer` / `TfidfVectorizer` and train a sentiment model (see project notebooks `02`–`08` for modeling).
